# Exercícios - Aula 6

Rodrigo Gimenes Rodrigues - CP3071782


## Exercício 1: Caçador de Outliers

Carregar o `tips` do Seaborn, achar outliers de `total_bill` pelo IQR, tirar esses pontos e comparar a média antes/depois.


In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np

df = sns.load_dataset('tips')

q1 = df['total_bill'].quantile(0.25)
q3 = df['total_bill'].quantile(0.75)
iqr = q3 - q1
lim_inf = q1 - 1.5 * iqr
lim_sup = q3 + 1.5 * iqr

print('Q1:', round(q1, 2), 'Q3:', round(q3, 2), 'IQR:', round(iqr, 2))
print('Limites:', round(lim_inf, 2), 'a', round(lim_sup, 2))

outliers = df[(df['total_bill'] < lim_inf) | (df['total_bill'] > lim_sup)]
print('Qtd outliers:', len(outliers))

df_sem = df[(df['total_bill'] >= lim_inf) & (df['total_bill'] <= lim_sup)].copy()

print('Média antes:', round(df['total_bill'].mean(), 2))
print('Média depois:', round(df_sem['total_bill'].mean(), 2))
print('Linhas antes/depois:', len(df), len(df_sem))

## Exercício 2: Scaling na Prática

DataFrame com altura (m) e peso (g). Aplicar MinMaxScaler e StandardScaler e ver a distância euclidiana entre dois pontos antes e depois.


In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from numpy.linalg import norm

df_pessoas = pd.DataFrame({
    'Altura': [1.60, 1.75, 1.82, 1.68],
    'Peso': [55000, 70000, 82000, 63000]
})
print(df_pessoas)

# distância entre as pessoas 0 e 1 (sem scaling)
p0 = df_pessoas.iloc[0].values
p1 = df_pessoas.iloc[1].values
dist_original = norm(p0 - p1)
print('Distância original (0 vs 1):', round(dist_original, 2))

minmax = MinMaxScaler()
df_minmax = pd.DataFrame(minmax.fit_transform(df_pessoas), columns=df_pessoas.columns)
print('
MinMaxScaler:')
print(df_minmax.round(3))
dist_minmax = norm(df_minmax.iloc[0].values - df_minmax.iloc[1].values)
print('Distância MinMax:', round(dist_minmax, 4))

std = StandardScaler()
df_std = pd.DataFrame(std.fit_transform(df_pessoas), columns=df_pessoas.columns)
print('
StandardScaler:')
print(df_std.round(3))
dist_std = norm(df_std.iloc[0].values - df_std.iloc[1].values)
print('Distância Standard:', round(dist_std, 4))

Sem scaling, o peso (em gramas) domina a distância porque a escala é bem maior que a da altura. Depois do MinMax ou StandardScaler as duas variáveis ficam comparáveis e a distância deixa de ser puxada só pelo peso.


## Exercício 3: Encoding Completo

Colunas: Cidade (SP, RJ, MG), Nível (Junior, Pleno, Senior) e Genero (M, F). Aplicar o encoding adequado e justificar.


In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

df_cat = pd.DataFrame({
    'Cidade': ['SP', 'RJ', 'MG', 'SP', 'RJ'],
    'Nivel': ['Junior', 'Pleno', 'Senior', 'Pleno', 'Junior'],
    'Genero': ['M', 'F', 'M', 'F', 'M']
})
print(df_cat)

# Cidade: nominal -> One-Hot
ohe_cidade = OneHotEncoder(sparse_output=False, dtype=int)
cidade_arr = ohe_cidade.fit_transform(df_cat[['Cidade']])
df_cidade = pd.DataFrame(cidade_arr, columns=ohe_cidade.get_feature_names_out(['Cidade']))

# Nivel: ordinal Junior < Pleno < Senior
ord_nivel = OrdinalEncoder(categories=[['Junior', 'Pleno', 'Senior']])
df_cat['Nivel_ord'] = ord_nivel.fit_transform(df_cat[['Nivel']]).astype(int)

# Genero: binária nominal -> 0/1 (ou One-Hot). Aqui usei mapa simples.
df_cat['Genero_bin'] = df_cat['Genero'].map({'F': 0, 'M': 1})

df_final = pd.concat([df_cidade, df_cat[['Nivel_ord', 'Genero_bin']]], axis=1)
print('
Depois do encoding:')
print(df_final)

Justificativa rápida:
- **Cidade:** categorias sem ordem (SP/RJ/MG) → One-Hot, pra não inventar ranking entre cidades.
- **Nível:** tem ordem natural Junior < Pleno < Senior → OrdinalEncoder.
- **Genero:** só duas classes nominais → 0/1 resolve. One-Hot também serviria; LabelEncoder “solto” eu evitei porque sugere ordem onde não tem.
